In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("MODEL DEPLOYMENT - 12-MONTH HOSPITALIZATION RISK PREDICTION")
print("="*80)

model_file = 'hospitalization_risk_model_gradient_boosting_balanced.pkl'

with open(model_file, 'rb') as f:
    model_package = pickle.load(f)

model = model_package['model']
scaler = model_package['scaler']
feature_cols = model_package['feature_columns']
model_name = model_package['model_name']
recommended_threshold = model_package['recommended_threshold']
performance = model_package['performance']

print(f"\nModel: {model_name}")
print(f"Recommended threshold: {recommended_threshold:.2f}")
print(f"Prediction window: 12-month (annual) hospitalization risk")

print("\n" + "="*80)
print("MODEL PERFORMANCE (FROM TRAINING)")
print("="*80)

print(f"\nTest Set Performance:")
print(f"  ROC-AUC: {performance['test_roc_auc']:.4f}")
print(f"  PR-AUC: {performance['test_pr_auc']:.4f}")
print(f"  Precision: {performance['test_precision']:.4f}")
print(f"  Recall: {performance['test_recall']:.4f}")
print(f"  F1 Score: {performance['test_f1_score']:.4f}")

print(f"\nCross-Validation Performance (Training Set):")
print(f"  Mean ROC-AUC: {performance['cv_mean']:.4f} (+/- {performance['cv_std']:.4f})")

print(f"\nValidation Set Performance:")
print(f"  ROC-AUC: {performance['validation_roc_auc']:.4f}")
print(f"  PR-AUC: {performance['validation_pr_auc']:.4f}")

print("\n" + "="*80)
print("THRESHOLD ANALYSIS")
print("="*80)

threshold_analysis = pd.DataFrame(model_package['threshold_analysis'])
print(f"\nAvailable thresholds and their trade-offs:\n")
print(threshold_analysis.to_string(index=False))

print(f"\nRecommended threshold: {recommended_threshold:.2f}")
print(f"This threshold was optimized on the validation set for best F1 score.")

print("\n" + "="*80)
print("PRODUCTION DEPLOYMENT CLASS")
print("="*80)

class HospitalizationRiskPredictor:
    
    def __init__(self, model_path, threshold=None):
        with open(model_path, 'rb') as f:
            package = pickle.load(f)
        
        self.model = package['model']
        self.scaler = package.get('scaler', None)
        self.feature_cols = package['feature_columns']
        self.model_name = package['model_name']
        self.threshold = threshold if threshold is not None else package['recommended_threshold']
        self.performance = package['performance']
        self.threshold_analysis = package.get('threshold_analysis', None)
    
    def predict(self, patient_data):
        if isinstance(patient_data, dict):
            patient_df = pd.DataFrame([patient_data])
        else:
            patient_df = patient_data.copy()
        
        missing_features = set(self.feature_cols) - set(patient_df.columns)
        if missing_features:
            raise ValueError(f"Missing required features: {missing_features}")
        
        X = patient_df[self.feature_cols]
        
        if self.scaler is not None:
            X = self.scaler.transform(X)
        
        probability = self.model.predict_proba(X)[0, 1]
        high_risk = int(probability >= self.threshold)
        
        if probability < 0.15:
            risk_level = 'Low Risk'
        elif probability < 0.25:
            risk_level = 'Moderate Risk'
        elif probability < 0.35:
            risk_level = 'High Risk'
        else:
            risk_level = 'Very High Risk'
        
        return {
            'probability': float(probability),
            'high_risk_flag': high_risk,
            'risk_level': risk_level,
            'model': self.model_name,
            'threshold': self.threshold
        }
    
    def batch_predict(self, dataframe):
        missing_features = set(self.feature_cols) - set(dataframe.columns)
        if missing_features:
            raise ValueError(f"Missing required features: {missing_features}")
        
        X = dataframe[self.feature_cols]
        
        if self.scaler is not None:
            X = self.scaler.transform(X)
        
        probabilities = self.model.predict_proba(X)[:, 1]
        
        results = dataframe.copy()
        results['hospitalization_probability'] = probabilities
        results['high_risk_flag'] = (probabilities >= self.threshold).astype(int)
        results['risk_level'] = pd.cut(
            probabilities,
            bins=[0, 0.15, 0.25, 0.35, 1.0],
            labels=['Low Risk', 'Moderate Risk', 'High Risk', 'Very High Risk']
        )
        
        return results
    
    def set_threshold(self, new_threshold):
        if not 0 <= new_threshold <= 1:
            raise ValueError(f"Threshold must be between 0 and 1, got {new_threshold}")
        
        self.threshold = new_threshold
        print(f"Threshold updated to: {new_threshold:.2f}")
        
        if self.threshold_analysis:
            matching = [t for t in self.threshold_analysis if abs(t['threshold'] - new_threshold) < 0.01]
            if matching:
                analysis = matching[0]
                print(f"Expected performance at this threshold:")
                print(f"  Precision: {analysis['precision']:.4f}")
                print(f"  Recall: {analysis['recall']:.4f}")
                print(f"  F1 Score: {analysis['f1_score']:.4f}")
                print(f"  % Population Flagged: {analysis['flagged_pct']:.1f}%")
    
    def get_feature_requirements(self):
        return self.feature_cols.copy()
    
    def get_model_info(self):
        return {
            'model_name': self.model_name,
            'threshold': self.threshold,
            'performance': self.performance,
            'n_features': len(self.feature_cols),
            'has_threshold_analysis': self.threshold_analysis is not None
        }

predictor = HospitalizationRiskPredictor(model_file)
print(f"\nPredictor initialized:")
print(f"  Model: {predictor.model_name}")
print(f"  Threshold: {predictor.threshold:.2f}")
print(f"  Features required: {len(predictor.feature_cols)}")

print("\n" + "="*80)
print("EXAMPLE: SINGLE PATIENT PREDICTION")
print("="*80)

example_patient = {
    'age': 78,
    'num_chronic_conditions': 4,
    'baseline_hospital_admits': 1.0,
    'baseline_er_visits': 0.5,
    'total_baseline_visits': 8.0,
    'gender_encoded': 1
}

for col in feature_cols:
    if col not in example_patient:
        example_patient[col] = 0

prediction = predictor.predict(example_patient)

print(f"\nPatient Profile:")
print(f"  Age: {example_patient['age']}")
print(f"  Chronic Conditions: {example_patient['num_chronic_conditions']}")
print(f"  Baseline Hospital Admits: {example_patient['baseline_hospital_admits']}")
print(f"  Baseline ER Visits: {example_patient['baseline_er_visits']}")

print(f"\nPrediction:")
print(f"  Hospitalization Probability: {prediction['probability']:.1%}")
print(f"  Risk Level: {prediction['risk_level']}")
print(f"  High Risk Flag: {'YES' if prediction['high_risk_flag'] else 'NO'}")

print("\n" + "="*80)
print("EXAMPLE: BATCH PREDICTIONS ON POPULATION")
print("="*80)

df = pd.read_csv('hospitalization_risk_data.csv')

from sklearn.preprocessing import LabelEncoder
if 'gender_encoded' not in df.columns and 'gender' in df.columns:
    df['gender_encoded'] = LabelEncoder().fit_transform(df['gender'])

predictions_df = predictor.batch_predict(df)

print(f"\nTotal patients: {len(predictions_df):,}")
print(f"Patients flagged as high-risk: {predictions_df['high_risk_flag'].sum():,} ({predictions_df['high_risk_flag'].mean()*100:.1f}%)")

print("\nRisk Stratification:")
risk_summary = predictions_df.groupby('risk_level').agg({
    'DESYNPUF_ID': 'count',
    'hospitalization_probability': 'mean'
}).round(3)

risk_summary.columns = ['Patient Count', 'Avg Probability']
risk_order = ['Low Risk', 'Moderate Risk', 'High Risk', 'Very High Risk']
risk_summary = risk_summary.reindex(risk_order)
print(risk_summary)

print("\nTop 10 Highest Risk Patients:")
top_risk = predictions_df.nlargest(10, 'hospitalization_probability')[
    ['DESYNPUF_ID', 'age', 'num_chronic_conditions', 'baseline_hospital_admits',
     'hospitalization_probability', 'risk_level']
].copy()

top_risk['hospitalization_probability'] = top_risk['hospitalization_probability'].apply(lambda x: f"{x:.1%}")
print(top_risk.to_string(index=False))

predictions_df.to_csv('patient_risk_predictions.csv', index=False)
print("\nPredictions saved to: patient_risk_predictions.csv")

print("\n" + "="*80)
print("ADJUSTING THRESHOLD EXAMPLE")
print("="*80)

print("\nCurrent threshold: 0.20")
print("Let's see what happens if we lower it to 0.15 for higher recall:\n")

predictor.set_threshold(0.15)

predictions_df_015 = predictor.batch_predict(df)
print(f"\nWith threshold 0.15:")
print(f"  Patients flagged: {predictions_df_015['high_risk_flag'].sum():,} ({predictions_df_015['high_risk_flag'].mean()*100:.1f}%)")

predictor.set_threshold(0.25)

predictions_df_025 = predictor.batch_predict(df)
print(f"\nWith threshold 0.25:")
print(f"  Patients flagged: {predictions_df_025['high_risk_flag'].sum():,} ({predictions_df_025['high_risk_flag'].mean()*100:.1f}%)")

predictor.set_threshold(recommended_threshold)
print(f"\nThreshold reset to recommended: {recommended_threshold:.2f}")

print("\n" + "="*80)
print("DEPLOYMENT COMPLETE")
print("="*80)


MODEL DEPLOYMENT - 12-MONTH HOSPITALIZATION RISK PREDICTION

Model: Gradient Boosting (Balanced)
Recommended threshold: 0.20
Prediction window: 12-month (annual) hospitalization risk

MODEL PERFORMANCE (FROM TRAINING)

Test Set Performance:
  ROC-AUC: 0.7558
  PR-AUC: 0.3440
  Precision: 0.2943
  Recall: 0.6710
  F1 Score: 0.4091

Cross-Validation Performance (Training Set):
  Mean ROC-AUC: 0.7544 (+/- 0.0059)

Validation Set Performance:
  ROC-AUC: 0.7504
  PR-AUC: 0.3434

THRESHOLD ANALYSIS

Available thresholds and their trade-offs:

 threshold  precision   recall  f1_score  flagged_pct
      0.10   0.233060 0.935210  0.373133    64.936404
      0.15   0.254610 0.835900  0.390329    53.128223
      0.20   0.284032 0.665959  0.398222    37.942592
      0.25   0.325722 0.484865  0.389671    24.089034
      0.30   0.380402 0.331917  0.354509    14.119972
      0.35   0.428090 0.202337  0.274793     7.648677
      0.40   0.459959 0.118959  0.189030     4.185287
      0.45   0.465587 0.0